# InstantSplat on Google Colab

少数の画像（2〜12枚）から3D Gaussian Splatting（PLY）を生成するノートブック。

**使い方:**
1. ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選択
2. 上から順にセルを実行
3. 画像をアップロード
4. 生成されたPLYをダウンロード

**2回目以降**: Google Driveにキャッシュ済みのため、セットアップは数十秒で完了します。

## 1. Google Drive マウント + GPU確認

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CACHE_DIR = '/content/drive/MyDrive/InstantSplat_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. InstantSplat セットアップ（Driveキャッシュ付き）

初回: クローン → ビルド → Driveに保存（10〜15分）  
2回目以降: Driveからコピー（数十秒）

In [ ]:
import os, shutil, time

CACHE_DIR = '/content/drive/MyDrive/InstantSplat_cache'
REPO_TAR = f'{CACHE_DIR}/instantsplat_repo.tar.gz'
CKPT_CACHE = f'{CACHE_DIR}/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth'
CUDA_MODULES_TAR = f'{CACHE_DIR}/cuda_modules.tar.gz'

os.chdir('/content')

# ========================================
# Step 1: Repository
# ========================================
if os.path.exists(REPO_TAR):
    print("[Cache] Driveからリポジトリを復元中...")
    start = time.time()
    !tar xzf "{REPO_TAR}" -C /content
    print(f"  復元完了 ({time.time()-start:.0f}秒)")
else:
    print("[初回] リポジトリをクローン中...")
    !git clone --recursive https://github.com/NVlabs/InstantSplat.git
    print("  Driveにキャッシュ保存中...")
    !tar czf "{REPO_TAR}" -C /content InstantSplat
    print("  保存完了")

os.chdir('/content/InstantSplat')

# ========================================
# Step 2: MASt3R Checkpoint (~1.3GB)
# ========================================
ckpt_dir = 'mast3r/checkpoints'
ckpt_file = f'{ckpt_dir}/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth'
os.makedirs(ckpt_dir, exist_ok=True)

if os.path.exists(CKPT_CACHE):
    if not os.path.exists(ckpt_file):
        print("[Cache] Driveからチェックポイントをコピー中...")
        start = time.time()
        shutil.copy(CKPT_CACHE, ckpt_file)
        print(f"  コピー完了 ({time.time()-start:.0f}秒)")
    else:
        print("[OK] チェックポイント既存")
else:
    print("[初回] チェックポイントをダウンロード中...")
    !wget -q --show-progress \
        https://download.europe.naverlabs.com/ComputerVision/MASt3R/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth \
        -P {ckpt_dir}
    print("  Driveにキャッシュ保存中...")
    shutil.copy(ckpt_file, CKPT_CACHE)
    print("  保存完了")

# ========================================
# Step 3: Python dependencies
# ========================================
print("\n[Pip] 依存パッケージをインストール中...")
!pip install -q roma evo opencv-python open3d trimesh plyfile tqdm scipy einops cython icecream imageio
!pip install -q "huggingface-hub[torch]>=0.22" gdown "pyglet<2" gradio

# ========================================
# Step 4: CUDA submodules (build is fast, cache the .so files)
# ========================================
if os.path.exists(CUDA_MODULES_TAR):
    print("\n[Cache] DriveからビルドCUDAモジュールを復元中...")
    !tar xzf "{CUDA_MODULES_TAR}" -C /
    # Still need to pip install so Python knows about them
    !pip install -q --no-build-isolation submodules/simple-knn 2>/dev/null
    !pip install -q --no-build-isolation submodules/diff-gaussian-rasterization 2>/dev/null
    !pip install -q --no-build-isolation submodules/fused-ssim 2>/dev/null
    print("  復元完了")
else:
    print("\n[初回] CUDAモジュールをビルド中（数分かかります）...")
    !pip install -q submodules/simple-knn
    !pip install -q submodules/diff-gaussian-rasterization
    !pip install -q submodules/fused-ssim
    # Cache the built modules
    print("  Driveにキャッシュ保存中...")
    # Find installed .so files
    import subprocess
    result = subprocess.run(['find', '/usr/local/lib/python3.10/dist-packages',
                             '-name', '*.so', '-path', '*/simple_knn*',
                             '-o', '-name', '*.so', '-path', '*/diff_gaussian*',
                             '-o', '-name', '*.so', '-path', '*/fused_ssim*'],
                            capture_output=True, text=True)
    so_files = [f for f in result.stdout.strip().split('\n') if f]
    # Also get the egg-info/dist-info dirs
    result2 = subprocess.run(['find', '/usr/local/lib/python3.10/dist-packages',
                              '-maxdepth', '1', '-type', 'd',
                              '-name', 'simple_knn*',
                              '-o', '-name', 'diff_gaussian*',
                              '-o', '-name', 'fused_ssim*'],
                             capture_output=True, text=True)
    dirs = [d for d in result2.stdout.strip().split('\n') if d]
    all_paths = ' '.join(so_files + dirs)
    if all_paths.strip():
        !tar czf "{CUDA_MODULES_TAR}" {all_paths}
        print("  保存完了")

# Optional: CRoPE
os.chdir('/content/InstantSplat/croco/models/curope')
!python setup.py build_ext --inplace 2>/dev/null || true
os.chdir('/content/InstantSplat')

# ========================================
# Step 5: MASt3Rモデル互換性パッチ
# （PyTorch新版でのcheckpoint読み込みエラーを修正）
# ========================================
model_py = '/content/InstantSplat/mast3r/model.py'
with open(model_py, 'r') as f:
    src = f.read()
old = "map_location='cpu')"
new = "map_location='cpu', weights_only=False)"
if old in src and new not in src:
    src = src.replace(old, new)
    with open(model_py, 'w') as f:
        f.write(src)
    print("\n[Patch] mast3r/model.py を修正しました（weights_only=False）")
else:
    print("\n[Patch] mast3r/model.py は修正済み")

print("\n" + "="*50)
print("セットアップ完了!")
print("="*50)

## 3. 画像をアップロード

同じ被写体/シーンを**異なる角度から撮影した2〜12枚**の画像をアップロードしてください。

**コツ:**
- 隣り合う写真のオーバーラップが多いほど良い（60%以上推奨）
- 照明条件は統一する
- ブレの少ないシャープな画像を使う

In [ ]:
import shutil
from google.colab import files

os.chdir('/content/InstantSplat')

# Scene name (change if you like)
SCENE_NAME = "my_scene"  # @param {type:"string"}

# Setup directories
source_path = f"/content/InstantSplat/assets/examples/{SCENE_NAME}/images"
os.makedirs(source_path, exist_ok=True)

# Clean previous uploads
for f in os.listdir(source_path):
    os.remove(os.path.join(source_path, f))

# Upload
print("画像ファイルを選択してください（複数選択可）:")
uploaded = files.upload()

for filename, data in uploaded.items():
    dst = os.path.join(source_path, filename)
    with open(dst, 'wb') as f:
        f.write(data)

n_images = len(os.listdir(source_path))
print(f"\n{n_images}枚の画像をアップロードしました: {source_path}")
!ls -la {source_path}

## 4. 3DGS 生成

3ステップで実行:
1. **init_geo** — MASt3Rで幾何推定（カメラ姿勢 + 点群の初期化）
2. **train** — Gaussian Splatting の最適化
3. **PLY保存** — 結果をPLYファイルとして出力

In [ ]:
os.chdir('/content/InstantSplat')

# --- Settings ---
N_VIEWS = n_images       # アップロードした画像枚数
TRAIN_ITER = 2000        # 学習イテレーション数（多いほど高品質、時間かかる）
IMAGE_SIZE = 512         # 入力画像リサイズ（大きいほど高品質だがVRAM消費大）

source = f"assets/examples/{SCENE_NAME}"
model = f"output/{SCENE_NAME}/{N_VIEWS}_views"

print(f"Scene: {SCENE_NAME}")
print(f"Views: {N_VIEWS}")
print(f"Iterations: {TRAIN_ITER}")
print(f"Image size: {IMAGE_SIZE}")
print(f"Output: {model}")

In [ ]:
%%time
os.chdir('/content/InstantSplat')

# Step 1: Geometry initialization with MASt3R
print("="*50)
print("Step 1/3: Geometry initialization (MASt3R)...")
print("="*50)

!python -W ignore ./init_geo.py \
    -s {source} \
    -m {model} \
    --n_views {N_VIEWS} \
    --image_size {IMAGE_SIZE} \
    --focal_avg \
    --co_vis_dsp \
    --conf_aware_ranking

print("\nGeometry initialization done.")

In [ ]:
%%time
os.chdir('/content/InstantSplat')

# Step 2: Train Gaussian Splatting
print("="*50)
print("Step 2/3: Training 3D Gaussian Splatting...")
print("="*50)

!python ./train.py \
    -s {source} \
    -m {model} \
    -r 1 \
    --n_views {N_VIEWS} \
    --iterations {TRAIN_ITER} \
    --save_iterations {TRAIN_ITER} \
    --pp_optimizer \
    --optim_pose

print("\nTraining done.")

In [ ]:
# Step 3: Find the output PLY
print("="*50)
print("Step 3/3: Locating output PLY...")
print("="*50)

import glob

ply_files = sorted(glob.glob(f"{model}/point_cloud/*/point_cloud.ply"))

if ply_files:
    output_ply = ply_files[-1]  # Latest iteration
    file_size = os.path.getsize(output_ply) / 1e6
    print(f"PLY found: {output_ply}")
    print(f"File size: {file_size:.1f} MB")
else:
    # Fallback: search all PLY files in output
    all_ply = sorted(glob.glob(f"{model}/**/*.ply", recursive=True))
    if all_ply:
        print("Found PLY files:")
        for p in all_ply:
            print(f"  {p} ({os.path.getsize(p)/1e6:.1f} MB)")
        output_ply = all_ply[-1]
    else:
        print("ERROR: No PLY file found. Check training logs above.")
        output_ply = None

## 5. PLYファイルをダウンロード

In [ ]:
if output_ply and os.path.exists(output_ply):
    # Rename for clarity
    download_name = f"{SCENE_NAME}_3dgs.ply"
    shutil.copy(output_ply, f"/content/{download_name}")
    print(f"Downloading {download_name}...")
    files.download(f"/content/{download_name}")
else:
    print("No PLY file to download.")

## 6. (オプション) レンダリング動画を生成

In [ ]:
%%time
os.chdir('/content/InstantSplat')

# Render interpolated video
print("Rendering interpolated video...")

!python ./render.py \
    -s {source} \
    -m {model} \
    -r 1 \
    --n_views {N_VIEWS} \
    --iterations {TRAIN_ITER} \
    --infer_video

# Find and download video
videos = sorted(glob.glob(f"{model}/**/*.mp4", recursive=True))
if videos:
    video_path = videos[-1]
    print(f"\nVideo: {video_path}")
    download_video = f"{SCENE_NAME}_render.mp4"
    shutil.copy(video_path, f"/content/{download_video}")
    files.download(f"/content/{download_video}")
else:
    print("No video generated.")

## 7. (オプション) Driveキャッシュを削除

不要になったらDriveの容量を解放できます（約1.6GB）。

In [ ]:
# Uncomment to delete cache
# import shutil
# shutil.rmtree('/content/drive/MyDrive/InstantSplat_cache', ignore_errors=True)
# print("Cache deleted.")

## Tips

### セットアップ時間の目安
| 回 | 時間 | 理由 |
|---|------|------|
| 初回 | 10〜15分 | クローン + ビルド + DL + Drive保存 |
| 2回目以降 | 30秒〜1分 | Driveからコピー + pip install |

### パラメータ調整
| パラメータ | デフォルト | 効果 |
|-----------|-----------|------|
| `TRAIN_ITER` | 2000 | 大きいほど高品質（3000-5000推奨）だが時間増 |
| `IMAGE_SIZE` | 512 | 大きいほど高精細だがVRAM増（T4なら512が安全） |

### 画像の撮り方
- 被写体の周りを**少しずつ角度を変えて**撮影
- 隣接する写真の**オーバーラップ60%以上**が理想
- 3〜6枚がバランス良い（多すぎるとVRAM不足の可能性）
- 照明は均一に、フラッシュは避ける

### 生成したPLYの活用
- **PLY Editor**: `ply-editor.html` で編集（密度調整、ワープ等）
- **MIDI Orchestra**: 3DGSとして読み込み可能（Trim 0%推奨）
- **SuperSplat**: https://superspl.at/editor で閲覧・編集

### Driveキャッシュの中身
| ファイル | サイズ |
|---------|--------|
| `instantsplat_repo.tar.gz` | ~200MB |
| `MASt3R_...metric.pth` | ~1.3GB |
| `cuda_modules.tar.gz` | ~100MB |
| **合計** | **~1.6GB** |